<a href="https://colab.research.google.com/github/alishbakhan21/mmash-sleep-hormones/blob/main/01_load_and_merge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question: Do saliva melatonin and cortisol differ between bedtime and waking, and do they relate to chronotype?

In [49]:
!wget -q https://physionet.org/static/published-projects/mmash/multilevel-monitoring-of-activity-and-sleep-in-healthy-people-1.0.0.zip -O mmash.zip

In [50]:
!ls -lh mmash.zip

-rw-r--r-- 1 root root 23M Mar  9  2024 mmash.zip


In [51]:
!unzip -q -o mmash.zip -d mmash_raw
!find mmash_raw -name "*.zip" -exec unzip -q -o {} -d mmash \;
!ls mmash

DataPaper


In [52]:
!ls mmash/*

user_1	 user_12  user_15  user_18  user_20  user_3  user_6  user_9
user_10  user_13  user_16  user_19  user_21  user_4  user_7
user_11  user_14  user_17  user_2   user_22  user_5  user_8


Data source. I used the MMASH dataset (Rossi et al., 2020, Data; PhysioNet, doi:10.13026/cerq-fc86). It contains 24-hour recordings from 22 healthy young adults, including saliva melatonin and cortisol sampled before sleep and after waking. The data is downloaded directly from PhysioNet and is not stored in this repository.

In [53]:
import pandas as pd, glob, os

def load_all(filename):
    frames = []
    for path in sorted(glob.glob(f"mmash/**/{filename}", recursive=True)):
        df = pd.read_csv(path, index_col=0)
        df["user"] = os.path.basename(os.path.dirname(path))
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

Loading. Each participant has their own folder. This function reads the same file from every folder, labels each row with the participant ID, and combines them into one table.

In [54]:
saliva = load_all("saliva.csv")
quest  = load_all("questionnaire.csv")
sleep  = load_all("sleep.csv")

for name, df in [("saliva", saliva), ("quest", quest), ("sleep", sleep)]:
    print(name, df.shape, df["user"].nunique(), "users")
    print(df.columns.tolist(), "\n")

saliva (42, 4) 21 users
['SAMPLES', 'Cortisol NORM', 'Melatonin NORM', 'user'] 

quest (22, 20) 22 users
['MEQ', 'STAI1', 'STAI2', 'Pittsburgh', 'Daily_stress', 'BISBAS_bis', 'BISBAS_reward', 'BISBAS_drive', 'BISBAS_fun', 'panas_pos_10', 'panas_pos_14', 'panas_pos_18', 'panas_pos_22', 'panas_pos_9+1', 'panas_neg_10', 'panas_neg_14', 'panas_neg_18', 'panas_neg_22', 'panas_neg_9+1', 'user'] 

sleep (22, 17) 21 users
['In Bed Date', 'In Bed Time', 'Out Bed Date', 'Out Bed Time', 'Onset Date', 'Onset Time', 'Latency', 'Efficiency', 'Total Minutes in Bed', 'Total Sleep Time (TST)', 'Wake After Sleep Onset (WASO)', 'Number of Awakenings', 'Average Awakening Length', 'Movement Index', 'Fragmentation Index', 'Sleep Fragmentation Index', 'user'] 



/tmp/ipykernel_462/3234969424.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True)


In [55]:
all_users = set(quest["user"])
print("No saliva:", all_users - set(saliva["user"]))
print("No sleep:", all_users - set(sleep["user"]))
print(sleep["user"].value_counts().head(3))
print(saliva.isna().sum())

No saliva: {'user_21'}
No sleep: {'user_11'}
user
user_1     2
user_10    1
user_12    1
Name: count, dtype: int64
SAMPLES           0
Cortisol NORM     0
Melatonin NORM    0
user              0
dtype: int64


Saliva: 21 of 22 participants (2 samples each). 21 has no saliva data.
Questionnaires: all 22 participants.
Sleep: 21 participants. 11 has no sleep data, and 1 has two sleep entries.

In [56]:
saliva.head(10)

,SAMPLES,Cortisol NORM,Melatonin NORM,user
0,before sleep,0.034137,1.738385e-08,user_1
1,wake up,0.077939,6.747893e-09,user_1
2,before sleep,0.037046,8.669611e-09,user_10
3,wake up,0.019659,2.573795e-09,user_10
4,before sleep,0.040592,2.038077e-09,user_11
5,wake up,0.015572,9.653476e-09,user_11
6,before sleep,0.155777,3.542804e-09,user_12
7,wake up,0.145431,8.639988e-09,user_12
8,before sleep,0.012278,1.902268e-09,user_13
9,wake up,0.034191,2.295933e-09,user_13


In [57]:
merged = saliva.merge(quest, on="user", how="left")
merged.head()

,SAMPLES,Cortisol NORM,Melatonin NORM,user,MEQ,STAI1,STAI2,Pittsburgh,Daily_stress,BISBAS_bis,...,panas_pos_10,panas_pos_14,panas_pos_18,panas_pos_22,panas_pos_9+1,panas_neg_10,panas_neg_14,panas_neg_18,panas_neg_22,panas_neg_9+1
0,before sleep,0.034137,1.738385e-08,user_1,47.0,41.0,43.0,5.0,23.0,22.0,...,21.0,17.0,12.0,18.0,17.0,11.0,13.0,13.0,10.0,10.0
1,wake up,0.077939,6.747893e-09,user_1,47.0,41.0,43.0,5.0,23.0,22.0,...,21.0,17.0,12.0,18.0,17.0,11.0,13.0,13.0,10.0,10.0
2,before sleep,0.037046,8.669611e-09,user_10,38.0,39.0,46.0,4.0,14.0,19.0,...,28.0,23.0,21.0,18.0,23.0,15.0,23.0,23.0,27.0,12.0
3,wake up,0.019659,2.573795e-09,user_10,38.0,39.0,46.0,4.0,14.0,19.0,...,28.0,23.0,21.0,18.0,23.0,15.0,23.0,23.0,27.0,12.0
4,before sleep,0.040592,2.038077e-09,user_11,38.0,36.0,43.0,7.0,17.0,23.0,...,23.0,16.0,22.0,30.0,13.0,15.0,15.0,11.0,13.0,14.0


Merge. I matched saliva samples to questionnaire answers by participant ID. Next step: reshape the data to one row per participant and decide how to handle missing and duplicate entries.

In [58]:
print(saliva["SAMPLES"].unique())

['before sleep' 'wake up']


In [59]:
mel = saliva.pivot(index="user", columns="SAMPLES", values="Melatonin NORM")
mel.head()

SAMPLES,before sleep,wake up
user,,
user_1,1.738385e-08,6.747893e-09
user_10,8.669611e-09,2.573795e-09
user_11,2.038077e-09,9.653476e-09
user_12,3.542804e-09,8.639988e-09
user_13,1.902268e-09,2.295933e-09


In [60]:
mel.columns = ["mel_bed", "mel_wake"]

In [61]:
cort = saliva.pivot(index="user", columns="SAMPLES", values="Cortisol NORM")
cort.columns = ["cort_bed", "cort_wake"]
cort.head()

,cort_bed,cort_wake
user,,
user_1,0.034137,0.077939
user_10,0.037046,0.019659
user_11,0.040592,0.015572
user_12,0.155777,0.145431
user_13,0.012278,0.034191


Reshaped the saliva data so each participant is one row, with separate bedtime and wake columns for melatonin and cortisol. Checked the column order against the sample labels before renaming.

In [62]:
sleep_main = (sleep.sort_values("Total Sleep Time (TST)", ascending=False)
                   .drop_duplicates("user"))
print(len(sleep_main), "users")

21 users


user 21 had two sleep periods; I kept the longer one as the main sleep episode.

In [63]:
data = (mel.join(cort)
           .join(quest.set_index("user")[["MEQ", "Pittsburgh"]])
           .join(sleep_main.set_index("user")[["Efficiency", "Total Sleep Time (TST)"]]))
data

,mel_bed,mel_wake,cort_bed,cort_wake,MEQ,Pittsburgh,Efficiency,Total Sleep Time (TST)
user,,,,,,,,
user_1,1.738385e-08,6.747893e-09,0.034137,0.077939,47.0,5.0,92.02,196
user_10,8.669611e-09,2.573795e-09,0.037046,0.019659,38.0,4.0,75.08,247
user_11,2.038077e-09,9.653476e-09,0.040592,0.015572,38.0,7.0,NaN,NaN
user_12,3.542804e-09,8.639988e-09,0.155777,0.145431,50.0,4.0,94.23,343
user_13,1.902268e-09,2.295933e-09,0.012278,0.034191,48.0,4.0,76.47,247
user_14,6.110973e-09,4.255579e-09,0.015788,0.039235,59.0,2.0,90.78,374
user_15,2.607041e-09,1.194220e-09,0.020579,0.042939,58.0,5.0,89.39,236
user_16,6.923590e-09,0.000000e+00,0.019456,0.042941,45.0,5.0,74.34,339
user_17,4.379331e-09,1.529949e-08,0.018304,0.095269,49.0,3.0,86.93,306


Combined saliva, questionnaire and sleep data into one table. 21 participants are included. 1 was dropped because they have no saliva data, and saliva is the main thing I'm looking at.

In [64]:
data.shape

(21, 8)

In [65]:
data["chronotype"] = pd.cut(data["MEQ"], bins=[15, 41, 58, 86],
                            labels=["evening", "intermediate", "morning"])
data["chronotype"].value_counts()

,count
chronotype,
intermediate,15
morning,4
evening,2


Using the standard MEQ cut-offs, I got 15 intermediate, 4 morning and 2 evening types. Only 2 evening types is too few to compare groups, so I'll use MEQ as a continuous score instead.

In [66]:
print(data.isna().sum())
data.describe()

mel_bed                   0
mel_wake                  0
cort_bed                  0
cort_wake                 0
MEQ                       0
Pittsburgh                0
Efficiency                1
Total Sleep Time (TST)    1
chronotype                0
dtype: int64


,mel_bed,mel_wake,cort_bed,cort_wake,MEQ,Pittsburgh,Efficiency
count,2.100000e+01,2.100000e+01,21.000000,21.000000,21.000000,21.000000,20.000000
mean,8.330234e-09,6.964888e-09,0.028053,0.069860,50.095238,5.380952,83.338000
std,6.707615e-09,6.392950e-09,0.030470,0.053220,7.119724,2.011870,6.781338
min,1.629907e-09,0.000000e+00,0.012017,0.015572,38.000000,2.000000,73.490000
25%,3.334482e-09,2.573795e-09,0.015166,0.041724,45.000000,4.000000,76.185000
50%,5.714439e-09,5.730512e-09,0.020206,0.057075,49.000000,5.000000,84.640000
75%,1.126009e-08,8.639988e-09,0.025968,0.081888,55.000000,7.000000,88.400000
max,2.396239e-08,2.853905e-08,0.155777,0.261252,64.000000,9.000000,94.230000


Some things I noticed:
user 16 has a wake melatonin value of exactly 0. That's probably below the detection limit rather than a real zero. I've kept it for now but flagged it.
Cortisol has a couple of very high values (bedtime: user 12, wake:user 2). I didn't remove them, and I'll use non-parametric tests so they don't distort the results.
User 11 has no sleep efficiency data.
Average cortisol is higher at waking than at bedtime, which fits the normal morning rise in cortisol. So the data looks sensible so far.
The average Pittsburgh score is about 5.4. Scores above 5 usually mean poor sleep quality, so a fair number of participants may be poor sleepers.

In [67]:
print(data[data["mel_wake"] == 0].index.tolist())
print(data["cort_bed"].idxmax(), data["cort_wake"].idxmax())
print(data[data["Efficiency"].isna()].index.tolist())

['user_16']
user_12 user_2
['user_11']
